#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the environment and installs required libraries for Spark-X2.6-1.7B.

import os
import subprocess
from IPython.display import clear_output

print("🚀 [@CoinNoin] Initializing Core Architecture...")
print("📦 [@CoinNoin] Installing Dependencies (This takes a moment)...")
# Install required transformers version and hf_transfer for ultra-fast downloads
!pip install -q transformers==4.57.1 accelerate hf_transfer

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import torch
from threading import Thread
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. Download and Load Spark-X2.6 Model
#@markdown The model will be fast-downloaded (if not cached) and loaded into the GPU's memory.

MODEL_NAME = "Spark-X2.6-1.7B"

MODEL_IDS = {
    "Spark-X2.6-1.7B": "XHToken/Spark-X2.6-1.7B"
}

model_id = MODEL_IDS[MODEL_NAME]
print(f"⚡ [@CoinNoin] Loading {model_id}...")
print(f"   CUDA available: {torch.cuda.is_available()}")

# Use fp16 on GPU, fp32 on CPU. Avoids bfloat16 issues on Colab T4 GPUs.
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"📥 [@CoinNoin] Fetching Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
)

print(f"📥 [@CoinNoin] Fetching and Loading Model into VRAM (Fast-transfer enabled)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

clear_output()
print(f"✅ [@CoinNoin] {MODEL_NAME} successfully loaded and ready for inference!")

In [ ]:
#@title 3. Spark-X2.6 Text Generation
#@markdown Enter your prompt and tweak the generation parameters. Because the model is already loaded in Cell 2, you can run this cell as many times as you want instantly.

PROMPT = "What is the capital of Germany?" #@param {type:"string"}
USE_THINKING = False #@param {type:"boolean"}
REASONING_EFFORT = "low" #@param ["low", "high"]

#@markdown ---
#@markdown ### ⚙️ Advanced Settings
MAX_NEW_TOKENS = 512 #@param {type:"slider", min:64, max:8192, step:64}
REPETITION_PENALTY = 1.0 #@param {type:"slider", min:1.0, max:2.0, step:0.05}
TEMPERATURE = "auto" #@param ["auto", "0.7", "0.9", "1.0"] {allow-input: true}
#@markdown *Leave TEMPERATURE on `auto` to use the model's recommended default (1.0), or type a custom float.*

import torch
from threading import Thread
from transformers import TextIteratorStreamer

print(f"\n\033[94m➜ [@CoinNoin] Generating | Model: {MODEL_NAME} | Thinking: {USE_THINKING} ({REASONING_EFFORT}) | Max Tokens: {MAX_NEW_TOKENS}\033[0m")

if TEMPERATURE == "auto":
    temperature = 1.0 if USE_THINKING else 0.7
else:
    temperature = float(TEMPERATURE)

top_p = 0.95
top_k = 0

messages = [
    {"role": "user", "content": PROMPT}
]

print("🧠 [@CoinNoin] Preparing prompt and thinking tokens...")
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=USE_THINKING,
    reasoning_effort=REASONING_EFFORT,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

streamer = TextIteratorStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True
)

generate_kwargs = dict(
    **inputs,
    streamer=streamer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=temperature > 0.0,
    temperature=temperature if temperature > 0.0 else 1.0,
    top_p=top_p,
    top_k=top_k,
    repetition_penalty=REPETITION_PENALTY,
)

print("⏳ [@CoinNoin] Generating response...\n")
print("=" * 80)
print(f"✨ [@CoinNoin] RESPONSE:\n")

thread = Thread(target=model.generate, kwargs=generate_kwargs)
thread.start()

for new_text in streamer:
    print(new_text, end="", flush=True)

thread.join()
print("\n" + "=" * 80)